In [1]:
# ============================================================
# 02_ssl_pretrain.ipynb
# SSL pretraining on single-receiver amplitude windows
# Default: strict inductive on source domain A only
# Optional: include B_finetune support windows only
# Never use target_eval_unseen_locations
# ============================================================

from pathlib import Path
import json
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# -----------------------------
# Config
# -----------------------------
DATA_ROOT = Path("/home/tonyliao/Location")
BUILD_DIR = DATA_ROOT / "dataset_build_hybrid"
OUT_DIR = DATA_ROOT / "ssl_pretrain_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# SSL policy
USE_B_FINETUNE_FOR_SSL = False   # keep False for strict inductive
USE_AB_JOINT_RETRAIN_FOR_SSL = False  # keep False; 04b does not imply SSL on AB_joint_retrain

# training
BATCH_SIZE = 64
EPOCHS = 200
LR = 1e-3

# input shaping
TARGET_T = 256
TARGET_S = None   # infer from first sample

# SimCLR
PROJ_DIM = 128
TEMPERATURE = 0.1

# augmentation
TIME_MASK_MAX_FRAC = 0.12
SC_MASK_MAX_FRAC = 0.12
NOISE_SIGMA = 0.02
GAIN_LOW = 0.9
GAIN_HIGH = 1.1

# -----------------------------
# Load splits
# -----------------------------
def load_npz(path: Path):
    obj = np.load(path, allow_pickle=True)
    return {k: obj[k] for k in obj.files}

A_train = load_npz(BUILD_DIR / "A_train.npz")
A_val   = load_npz(BUILD_DIR / "A_val.npz")

ssl_paths = list(A_train["amp_path"].astype(str))
ssl_sources = ["A_train"]

if USE_B_FINETUNE_FOR_SSL:
    B_finetune = load_npz(BUILD_DIR / "B_finetune.npz")
    ssl_paths += list(B_finetune["amp_path"].astype(str))
    ssl_sources.append("B_finetune")

if USE_AB_JOINT_RETRAIN_FOR_SSL:
    # disabled by default; included only if explicitly requested
    AB_joint_retrain = load_npz(BUILD_DIR / "AB_joint_retrain.npz")
    ssl_paths += list(AB_joint_retrain["amp_path"].astype(str))
    ssl_sources.append("AB_joint_retrain")

# validation remains source-only by default
ssl_val_paths = list(A_val["amp_path"].astype(str))

print("SSL train windows:", len(ssl_paths))
print("SSL val windows:", len(ssl_val_paths))
print("SSL sources:", ssl_sources)

if len(ssl_paths) == 0:
    raise ValueError("No SSL training windows found.")

if len(ssl_val_paths) == 0:
    raise ValueError("No SSL validation windows found.")

# -----------------------------
# Helpers
# -----------------------------
def ensure_3d(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 2:
        x = x[..., None]
    return x

def resize_to_target(x, target_t, target_s):
    x_tf = tf.convert_to_tensor(x, dtype=tf.float32)
    x_tf = tf.image.resize(x_tf, size=(target_t, target_s), method="bilinear")
    return x_tf.numpy().astype(np.float32)

def zscore_per_sample(x):
    mu = np.mean(x, axis=(0, 1), keepdims=True)
    sd = np.std(x, axis=(0, 1), keepdims=True) + 1e-6
    return ((x - mu) / sd).astype(np.float32)

def load_amp(path):
    x = np.load(str(path)).astype(np.float32)
    x = ensure_3d(x)
    return x

sample = load_amp(ssl_paths[0])
if TARGET_S is None:
    TARGET_S = sample.shape[1]

print("TARGET_T =", TARGET_T)
print("TARGET_S =", TARGET_S)
print("sample shape =", sample.shape)

def preprocess_amp(path):
    x = load_amp(path)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

# -----------------------------
# Augmentations for same-window SSL
# -----------------------------
def random_time_mask(x, max_frac=0.12):
    x = x.copy()
    T = x.shape[0]
    w = max(1, int(T * np.random.uniform(0.04, max_frac)))
    s = np.random.randint(0, max(1, T - w + 1))
    x[s:s+w, :, :] = 0.0
    return x

def random_subcarrier_mask(x, max_frac=0.12):
    x = x.copy()
    S = x.shape[1]
    w = max(1, int(S * np.random.uniform(0.04, max_frac)))
    s = np.random.randint(0, max(1, S - w + 1))
    x[:, s:s+w, :] = 0.0
    return x

def random_gain(x, low=0.9, high=1.1):
    g = np.random.uniform(low, high)
    return (x * g).astype(np.float32)

def random_noise(x, sigma=0.02):
    return (x + np.random.normal(0, sigma, size=x.shape)).astype(np.float32)

def augment_amp(x):
    y = x.copy()
    if np.random.rand() < 0.8:
        y = random_time_mask(y, max_frac=TIME_MASK_MAX_FRAC)
    if np.random.rand() < 0.8:
        y = random_subcarrier_mask(y, max_frac=SC_MASK_MAX_FRAC)
    if np.random.rand() < 0.8:
        y = random_gain(y, low=GAIN_LOW, high=GAIN_HIGH)
    if np.random.rand() < 0.8:
        y = random_noise(y, sigma=NOISE_SIGMA)
    y = zscore_per_sample(y)
    return y.astype(np.float32)

# -----------------------------
# SSL dataset
# -----------------------------
class SSLSequence(keras.utils.Sequence):
    def __init__(self, amp_paths, batch_size=32, shuffle=True):
        self.amp_paths = np.asarray(amp_paths, dtype=object)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.amp_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        inds = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        x1, x2 = [], []

        for i in inds:
            x = preprocess_amp(self.amp_paths[i])
            v1 = augment_amp(x)
            v2 = augment_amp(x)
            x1.append(v1)
            x2.append(v2)

        return (
            np.stack(x1).astype(np.float32),
            np.stack(x2).astype(np.float32),
        )

train_seq = SSLSequence(ssl_paths, batch_size=BATCH_SIZE, shuffle=True)
val_seq   = SSLSequence(ssl_val_paths, batch_size=BATCH_SIZE, shuffle=False)

bx1, bx2 = train_seq[0]
print("view1:", bx1.shape, bx1.dtype)
print("view2:", bx2.shape, bx2.dtype)

# -----------------------------
# Encoder
# -----------------------------
def conv_block(x, filters, pool=True, dropout=0.0):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    if pool:
        x = layers.MaxPooling2D(pool_size=2)(x)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)
    return x

def build_encoder(input_shape, proj_dim=128):
    inp = keras.Input(shape=input_shape, name="amp_ssl_input")

    x = conv_block(inp, 32, pool=True, dropout=0.05)
    x = conv_block(x, 64, pool=True, dropout=0.05)
    x = conv_block(x, 128, pool=True, dropout=0.10)
    x = conv_block(x, 256, pool=True, dropout=0.10)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    embedding = layers.Dense(256, activation="relu", name="embedding")(x)

    proj = layers.Dense(256, activation="relu")(embedding)
    proj = layers.Dense(proj_dim, name="projection")(proj)

    encoder = keras.Model(inp, embedding, name="amp_ssl_encoder")
    projector = keras.Model(inp, proj, name="amp_ssl_projector")
    return encoder, projector

input_shape = bx1.shape[1:]
encoder, projector = build_encoder(input_shape, proj_dim=PROJ_DIM)
encoder.summary()

# -----------------------------
# NT-Xent / SimCLR loss
# -----------------------------
def l2_normalize(x):
    return tf.math.l2_normalize(x, axis=1)

def nt_xent_loss(z1, z2, temperature=0.1):
    z1 = l2_normalize(z1)
    z2 = l2_normalize(z2)

    batch_size = tf.shape(z1)[0]
    z = tf.concat([z1, z2], axis=0)  # [2N, d]
    sim = tf.matmul(z, z, transpose_b=True) / temperature

    # remove self-similarity
    mask = tf.eye(2 * batch_size)
    sim = sim - mask * 1e9

    pos_idx = tf.concat([
        tf.range(batch_size, 2 * batch_size),
        tf.range(0, batch_size)
    ], axis=0)

    labels = tf.one_hot(pos_idx, depth=2 * batch_size)
    loss = tf.keras.losses.categorical_crossentropy(labels, sim, from_logits=True)
    return tf.reduce_mean(loss)

# -----------------------------
# Custom training model
# -----------------------------
class SimCLRTrainer(keras.Model):
    def __init__(self, projector, temperature=0.1):
        super().__init__()
        self.projector = projector
        self.temperature = temperature
        self.loss_tracker = keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def train_step(self, data):
        x1, x2 = data
        with tf.GradientTape() as tape:
            z1 = self.projector(x1, training=True)
            z2 = self.projector(x2, training=True)
            loss = nt_xent_loss(z1, z2, temperature=self.temperature)

        grads = tape.gradient(loss, self.projector.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.projector.trainable_variables))
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        x1, x2 = data
        z1 = self.projector(x1, training=False)
        z2 = self.projector(x2, training=False)
        loss = nt_xent_loss(z1, z2, temperature=self.temperature)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

trainer = SimCLRTrainer(projector, temperature=TEMPERATURE)
trainer.compile(optimizer=keras.optimizers.Adam(LR))

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(OUT_DIR / "amp_ssl_projector_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(OUT_DIR / "ssl_pretrain_log.csv")),
]

history = trainer.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

# -----------------------------
# Save final models
# -----------------------------
trainer.projector.save(OUT_DIR / "amp_ssl_projector_final.keras")

# best encoder snapshot from current best-weight trainer state
amp_ssl_encoder_best = keras.Model(
    trainer.projector.input,
    trainer.projector.get_layer("embedding").output,
    name="amp_ssl_encoder"
)
amp_ssl_encoder_best.save(OUT_DIR / "amp_ssl_encoder_best.keras")

# final encoder snapshot
amp_ssl_encoder_final = keras.Model(
    trainer.projector.input,
    trainer.projector.get_layer("embedding").output,
    name="amp_ssl_encoder"
)
amp_ssl_encoder_final.save(OUT_DIR / "amp_ssl_encoder_final.keras")

with open(OUT_DIR / "ssl_pretrain_history.json", "w", encoding="utf-8") as f:
    json.dump(history.history, f, indent=2)

summary = {
    "protocol": "single_receiver_ssl",
    "strict_inductive": (not USE_B_FINETUNE_FOR_SSL) and (not USE_AB_JOINT_RETRAIN_FOR_SSL),
    "use_B_finetune_for_ssl": USE_B_FINETUNE_FOR_SSL,
    "use_AB_joint_retrain_for_ssl": USE_AB_JOINT_RETRAIN_FOR_SSL,
    "ssl_sources": ssl_sources,
    "used_target_eval_unseen_locations": False,
    "train_windows": int(len(ssl_paths)),
    "val_windows": int(len(ssl_val_paths)),
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "target_t": TARGET_T,
    "target_s": int(TARGET_S),
    "projection_dim": PROJ_DIM,
    "temperature": TEMPERATURE,
}

with open(OUT_DIR / "ssl_pretrain_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved to:", OUT_DIR)
print(json.dumps(summary, indent=2))

import gc
gc.collect()
tf.keras.backend.clear_session()
print("Session cleared.")

2026-04-13 13:46:43.522740: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-13 13:46:43.529110: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776088003.536570 1563497 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776088003.538821 1563497 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776088003.544623 1563497 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

SSL train windows: 2969
SSL val windows: 742
SSL sources: ['A_train']
TARGET_T = 256
TARGET_S = 41
sample shape = (256, 41, 2)


I0000 00:00:1776088004.750125 1563497 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22181 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


view1: (64, 256, 41, 2) float32
view2: (64, 256, 41, 2) float32


Model: "amp_ssl_encoder"

...

47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - loss: 1.8129

2026-04-13 13:47:09.913270: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_469', 120 bytes spill stores, 120 bytes spill loads




Epoch 1: val_loss improved from inf to 3.36623, saving model to /home/tonyliao/Location/ssl_pretrain_runs/amp_ssl_projector_best.keras
47/47 ━━━━━━━━━━━━━━━━━━━━ 25s 321ms/step - loss: 1.8006 - val_loss: 3.3662 - learning_rate: 0.0010
Epoch 2/200


/home/tonyliao/tensorflow_env/lib/python3.10/site-packages/keras/src/saving/saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)


47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 0.7318
Epoch 2: val_loss improved from 3.36623 to 2.48227, saving model to /home/tonyliao/Location/ssl_pretrain_runs/amp_ssl_projector_best.keras
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 169ms/step - loss: 0.7310 - val_loss: 2.4823 - learning_rate: 0.0010
Epoch 3/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - loss: 0.6161
Epoch 3: val_loss improved from 2.48227 to 2.01187, saving model to /home/tonyliao/Location/ssl_pretrain_runs/amp_ssl_projector_best.keras
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 171ms/step - loss: 0.6158 - val_loss: 2.0119 - learning_rate: 0.0010
Epoch 4/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 0.5642
Epoch 4: val_loss improved from 2.01187 to 1.83511, saving model to /home/tonyliao/Location/ssl_pretrain_runs/amp_ssl_projector_best.keras
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 168ms/step - loss: 0.5640 - val_loss: 1.8351 - learning_rate: 0.0010
Epoch 5/200
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - loss: 0.5353
Epoch 5: val_loss improved